# Reproducing Gao et al. (2025) from dandiset 001199

Paper. Gao H, Ramachandran S, Yu K, He B (2025). Transcranial Focused Ultrasound Modulates
Feedforward and Feedback Cortico-Thalamo-Cortical Pathways by Selectively Activating Excitatory
Neurons. *J Neurosci* 45(23):e2218242025.
[doi:10.1523/jneurosci.2218-24.2025](https://doi.org/10.1523/jneurosci.2218-24.2025). The claims
file was written from the version of record in PubMed Central (PMC12139594), with its Methods and
all figure legends; the paper has no supplement.

Dandiset. [DANDI:001199](https://dandiarchive.org/dandiset/001199/0.250829.1250), version
0.250829.1250. The paper does not cite this dandiset: its Methods name dandiset 000946 as the
repository of all its data, with no version. 001199 has four published versions (0.250712.1551,
0.250713.1101, 0.250828.1941, 0.250829.1250). The cell below shows that all four and the draft
hold the same 229 files (same paths and blobs) and differ only in metadata the analysis does not
use (the order of contributors). The most recent published version is pinned, as the protocol
says when the paper cites none.

What the dandiset holds. Sorted units (spike times only) and trial tables for 13 rats, recorded
in August and September 2024 (4 rats) and February to July 2025 (9 rats). There are no spike
waveforms and no RSU or FSU labels, every session uses PD 200 us, and AC recordings come from 7 rats.

Claims. The claims file (`claims/001199/10.1523-jneurosci.2218-24.2025.yaml`) has 35 claims.

| Claim | Tested here | Why not |
|---|---|---|
| subjects | yes | |
| fig11b-ac-responses | yes | |
| fig2-target-artifact | no | raw voltage and target-determination trials not in the dandiset |
| fig4-unit-counts, fig5-unit-counts, fig6-unit-counts | no | RSU and FSU types need waveforms, not in the dandiset |
| fig4-timelocked-rsu-3000, fig4-postsonication-3000, fig4-30hz-responses, fig4-no-fsu-change | no | as above (types) |
| fig5a-sham-no-response, fig5b-cg1-control-no-response | no | types; and no CG1 session at 3,000 Hz |
| fig6a-s1-rsu-timelocked-high-prf, fig6a-s1-rsu-delayed, fig6b-s1-fsu-prf, fig6c-pom-rsu-prf, fig6d-pom-fsu-prf | no | types |
| fig7-9-unit-counts, fig7a to fig7d, fig8a, fig8b, fig9a, fig9b | no | the DC series (PDs other than 200 us) are not in the dandiset; and types |
| fig10a-dsttc-3000-during, fig10a-dsttc-3000-post, fig10b-dsttc-30, fig10c-sttc-constant-pd | no | types |
| fig10d-sttc-60dc, fig10e-sttc-30hz-dc, fig10f-sttc-3000-dc | no | DC series missing; and types |
| fig11c-ac-to-s1-30hz, fig11d-ac-to-s1-3000 | no | S1 RSU and FSU types |

Every file is streamed from the archive; nothing is written to disk.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from scipy import stats

import tfus

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
assets = tfus.list_assets()
versions = ["0.250712.1551", "0.250713.1101", "0.250828.1941", "0.250829.1250", "draft"]
pinned = set(zip(assets.path, assets.blob))
for v in versions:
    a = tfus.list_assets(v)
    print(f"version {v}: {len(a)} assets, {a['size'].sum()} bytes, "
          f"(path, blob) pairs differing from the pinned version: {len(pinned ^ set(zip(a.path, a.blob)))}")
m_first, m_pin = tfus.version_metadata(versions[0]), tfus.version_metadata()
print("metadata fields that differ between 0.250712.1551 and the pinned version:",
      sorted(k for k in set(m_first) | set(m_pin) if m_first.get(k) != m_pin.get(k)))
print("\nDandiset description (pinned version):\n", m_pin["description"])
print("\nassetsSummary numberOfSubjects:", m_pin["assetsSummary"]["numberOfSubjects"],
      "; variableMeasured:", m_pin["assetsSummary"]["variableMeasured"])

In [ ]:
started = time.time()
files = tfus.read_many(assets.asset_id.tolist(), spikes=True)
print(f"read {len(files)} files from dandiset {tfus.DANDISET_ID} version {tfus.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

rows = []
for (_, a), f in zip(assets.iterrows(), files):
    c = tfus.parse_condition(f["identifier"], f["notes"])
    lab = f["celltype_label"]
    rows.append({
        "path": a.path, "rat": f["subject_id"], "start": f["session_start"][:19], "identifier": f["identifier"],
        "notes": f["notes"], "data_collection": f["data_collection"], "device": f["device"],
        "n_electrodes": f["n_electrodes"], "n_units": f["n_units"], "n_trials": len(f["trial_start"]),
        "label_values": "" if lab is None else ",".join(sorted(set(lab))),
        "n_pass": sum(len(s) > tfus.MIN_SPIKES for s in f["spikes"]), **c})
info = pd.DataFrame(rows)
info["cohort"] = np.where(info.rat.isin(["BH590", "BH594", "BH596", "BH615"]), "2024", "2025")

## Data check

Each check of the description against the values is stated under its output.

In [ ]:
print("Files per rat and cohort (cohort by the identifier format; see tfus.py):")
print(info.groupby(["cohort", "rat"]).agg(files=("path", "size"), first=("start", "min"), last=("start", "max"),
                                          units=("n_units", "sum"), units_over_2000=("n_pass", "sum")).to_string())
print("\nDistinct rats:", info.rat.nunique(), "; subject age field values:", sorted({f['age'] for f in files}),
      "; sex:", sorted({f['sex'] for f in files}))
print("\nDevices by cohort and recorded area (files):")
print(pd.crosstab([info.cohort, info.rec], info.device).T.to_string())
print("Electrodes per file:", info.n_electrodes.value_counts().to_dict(),
      "; electrode locations:", sorted({tuple(f['electrode_locations']) for f in files}))
print("\nUnits table columns:", pd.Series([",".join(f["units_columns"]) for f in files]).value_counts().to_dict())
print("celltype_label description:", sorted({f["units_descriptions"].get("celltype_label", "") for f in files} - {""}))
print("celltype_label values per file:", info.label_values.value_counts().to_dict())

The archive's summary says 13 subjects, as the files hold. Every file's age is P24W. The units
tables hold spike times and, in 96 files, a `celltype_label` column described as "label 1 = RSU,
label 2 = FSU". No file holds a 1 or a 2: the column holds 0 in the files of BH590 and BH594 and
the area of the unit (S1 or POm) in those of BH596 and BH615. The 133 files of the 2025 cohort
have no such column. No file holds waveforms. So the dandiset gives no way to type units as RSU or
FSU, which the paper does from the waveforms (Methods, Neuronal type classification). The
description says S1 is recorded with 64-channel probes and POm with 32-channel probes, and the
files agree: S1 files list A1x64-Edge probes (four BH590 files a device "H9"), the POm and AC files
of the 2025 cohort the A1x32-Poly3, and the 2024 files of BH596 and BH615 both. The paper used
two 32-channel A1x32-Poly3 probes. Electrode locations are all "unknown".

In [ ]:
print("Conditions (recording area, target, PRF, voltage written in the identifier):")
print(info.groupby(["cohort", "rec", "target", "prf", "volt"], dropna=False).size().rename("files").to_string())
print("\nPD values in identifiers:", info.pd.value_counts(dropna=False).to_dict())
print("data_collection values:", info.data_collection.value_counts().to_dict())
print("\nIdentifier tags:", info.tags.value_counts().to_dict())

Every stimulated file has PD 200 us, so only the constant-PD series of the paper (Fig. 4, 6) is
here; the duty-cycle series of Fig. 7 to 9 (PDs of 2 to 30,000 us) is not. Voltage is written only
in the 2024 identifiers (5, 10 or 15 V); the paper used 5 V (88 kPa in situ). The
`data_collection` field gives "PRESSURE:67V" for most 2024 files whatever the identifier says (the
67 is the ultrasound duration), so it does not state the pressure. In the 2025 files neither field
gives a voltage. Targets other than S1 and sham appear: POM (stimulation at POm, not in the paper),
CG1 (the paper's control, here at 300 and 4,500 Hz, where the paper used 3,000 Hz), and "S1-6",
"NEW", "y-1" and "SEP", which nothing in the dandiset explains.

In [ ]:
# Trials: counts, durations, intersonication intervals (Methods: 500 trials, ISI 2.5 s with 10% jitter).
iti = np.concatenate([np.diff(f["trial_start"]) for f in files])
dur = np.concatenate([f["trial_stop"] - f["trial_start"] for f in files])
print("Trials per file:", info.n_trials.value_counts().sort_index().to_dict())
print("Trial duration (stop - start), s: min %.4f max %.4f" % (dur.min(), dur.max()))
print("Interval between trial starts, s: percentiles 0, 1, 50, 99, 100 =", np.round(np.percentile(iti, [0, 1, 50, 99, 100]), 3))
spk = np.concatenate([np.concatenate(f["spikes"]) for f in files if f["n_units"]])
print("Spike times, s: min %.3f max %.1f" % (spk.min(), spk.max()))
uns = info.assign(n_unsorted=[f["n_unsorted"] for f in files])
print("Units whose spike times are not in order, as stored: %d in %d files (sorted on reading):"
      % (uns.n_unsorted.sum(), (uns.n_unsorted > 0).sum()), uns[uns.n_unsorted > 0].groupby("rat").n_unsorted.sum().to_dict())

# Recordings deposited twice: identical spike trains, and files sharing a trial table.
info["spike_key"] = [hash(np.concatenate(f["spikes"]).round(6).tobytes()) if f["n_units"] else None for f in files]
info["trial_key"] = [hash(np.round(f["trial_start"], 5).tobytes()) for f in files]
dup = info[info.duplicated("spike_key", keep=False) & info.spike_key.notna()]
print("\nFiles with identical spike trains:")
print(dup[["path", "identifier", "notes", "n_units"]].to_string(index=False))
shared = info[info.duplicated("trial_key", keep=False)].sort_values(["trial_key", "path"])
print(f"\n{len(shared)} files share their trial table with another file, in "
      f"{shared.trial_key.nunique()} groups; areas per group:")
grp = shared.groupby("trial_key").agg(rat=("rat", "first"), areas=("rec", lambda x: "+".join(sorted(x))),
                                      targets=("target", lambda x: "/".join(sorted(set(x)))),
                                      prf=("prf", "first"), starts=("start", lambda x: " ".join(sorted(x))))
print(grp.sort_values(["rat", "prf"]).to_string(index=False))

Trials number 499 in most files (the paper: 500), and fewer in some (113 to 480); one 2024 baseline
has 400 and two 2025 sham files 149. Each trial lasts 2.2 s, and trial starts are 2.25 to 2.75 s
apart, the 2.5 s with 10% jitter of the Methods. In 64 units of 16 files of the 2025 cohort the
stored spike times are not in increasing order (one to seven places per unit), which breaks the
NWB convention; they are sorted on reading.

Two files hold the same spike trains and trial table: BH646 `ses-None` and `ses-20250308T003900`,
both AC at 30 Hz. The first is counted once, as the second. In the 2025 cohort files of two areas
share a trial table, which shows that they are the two probes of one session: S1 with AC, or S1
with POm, matching the paper's two areas per session. The session start times of such a pair differ
by minutes, and for BH608 the two sham files (POm, S1) share a trial table though their start times
are five months apart (2025-02-06 and 2025-07-11), so `session_start_time` does not date a session
in this cohort. In the BH608 30 Hz pair the S1 file says stimulation at S1 and the POm file
stimulation at "SEP". In the 2024 cohort each file is one session; BH596 and BH615 files hold S1
and POm units (by `celltype_label`), BH590 and BH594 files S1 only.

In [ ]:
info_u = info[info.path != "sub-BH646/sub-BH646_ses-None_ecephys.nwb"].copy()   # the duplicate, counted once
keep = set(info_u.index)


def unit_area(i, k):
    f, r = files[i], info.loc[i]
    if r.cohort == "2024":
        lab = f["celltype_label"][k]
        return lab if lab in ("S1", "POm") else "S1"   # BH590/BH594 files: S1 probe only (notes, device)
    return r.rec


# Clock check: fraction of units (more than 2,000 spikes) whose count in 0-400 ms after the trial start
# differs from 400 ms before it (paired Wilcoxon, p < 0.01), with the true starts and with the starts
# shifted by half an intersonication interval.
rows = []
for i in keep:
    f, r = files[i], info.loc[i]
    sp = [s for s in f["spikes"] if len(s) > tfus.MIN_SPIKES]
    if not sp:
        continue
    on = f["trial_start"][(f["trial_start"] > 1.5) & (f["trial_start"] < f["trial_start"].max() - 1.5)]
    rows.append({"cohort": r.cohort, "rec": r.rec, "target": r.target, "rat": r.rat, "prf": r.prf, "units": len(sp),
                 "true": tfus.modulation_fraction(sp, on), "shifted": tfus.modulation_fraction(sp, on + tfus.SHIFT_S)})
clock = pd.DataFrame(rows)
clock["kind"] = np.where(clock.target == "none", "sham", "stimulated")
print("Fraction of modulated units, mean over files:")
print(clock.groupby(["cohort", "kind", "rec"]).agg(files=("units", "size"), units=("units", "sum"),
                                                   true=("true", "mean"), shifted=("shifted", "mean")).round(3).to_string())
for (c, k), g in clock.groupby(["cohort", "kind"]):
    if len(g) > 5:
        print(f"{c} {k}: true above shifted in {(g.true > g.shifted).sum()} of {len(g)} files; "
              f"paired Wilcoxon p = {stats.wilcoxon(g.true, g.shifted).pvalue:.2g}")

In [ ]:
# Per-file population PSTH (sum over units with more than 2,000 spikes, % of the file's mean), 25 ms bins.
bins = np.arange(-0.2, 0.6001, 0.025)


def pop_psth(i):
    f = files[i]
    tot = np.zeros(len(bins) - 1)
    for s in f["spikes"]:
        if len(s) > tfus.MIN_SPIKES:
            for t in f["trial_start"]:
                tot += np.histogram(s - t, bins)[0]
    return tot / tot.mean() * 100 if tot.sum() else tot


show = info_u[(info_u.n_pass > 0) & (((info_u.rat == "BH615") & info_u.identifier.str.contains("300_200_67_10_S1|3000_200_67_15_S1$"))
               | ((info_u.rat == "BH648") & (info_u.prf == 4500)) | ((info_u.rat == "BH594") & info_u.identifier.str.contains("coating_3000_200_67_10$|coating_BASELINE")))]
np.set_printoptions(linewidth=220, precision=0, suppress=True)
print("bin starts (ms):", np.round(bins[:-1] * 1000).astype(int))
psth_rows = {}
for i, r in show.iterrows():
    key = f"{r.rat} {r.identifier} ({r.rec})"
    psth_rows[key] = pop_psth(i)
    print(f"{key:40s}", psth_rows[key])
fig, ax = plt.subplots(figsize=(9, 3.5))
for k, v in psth_rows.items():
    ax.plot(bins[:-1] * 1000, v, label=k, drawstyle="steps-post")
ax.axvspan(0, 67, color="0.9")
ax.set_xlabel("time from trial start (ms)"); ax.set_ylabel("population rate (% of mean)"); ax.legend(fontsize=7)
ax.set_yscale("log")
print("plotted range: %.0f to %.0f %%" % (min(v.min() for v in psth_rows.values()), max(v.max() for v in psth_rows.values())))

In the 2024 cohort the trial starts are the sonication onsets, as far as the spikes show. The
fraction of units modulated in the 400 ms after a trial start is 0.39 on average, against 0.13
with starts shifted by 1.25 s (true above shifted in 50 of 92 stimulated files), and 0 in the sham
files. Two BH615
sessions (300 Hz at 10 V, 3,000 Hz at 15 V) show a burst of detected spikes confined to the first
three 25 ms bins (0 to 75 ms), the 67 ms of sonication, which looks like an artefact locked to the
ultrasound. Other files show the paper's delayed increase, from about 125 to 200 ms.

In the 2025 cohort, which holds all the AC recordings, this could not be established for most
sessions. Its files show no more modulation with the true starts than with shifted ones (true
above shifted in 7 of 108 stimulated files). The one
exception is BH648 at 4,500 Hz, where the S1 population rises from about 200 ms after the trial
start, the delay of the paper's postsonication responses. Its AC file shares that trial table. For
the other 2025 sessions the outputs cannot tell missing responses from trial times that are not
the sonication onsets.

In [ ]:
# Units with more than 2,000 spikes, by area and condition (untyped), against the paper's typed totals.
rows = []
for i in keep:
    r = info.loc[i]
    for k, s in enumerate(files[i]["spikes"]):
        if len(s) > tfus.MIN_SPIKES:
            rows.append({"cohort": r.cohort, "rat": r.rat, "area": unit_area(i, k), "target": r.target,
                         "prf": r.prf, "volt": r.volt, "file": i})
units = pd.DataFrame(rows)
print("Units with more than 2,000 spikes:", len(units), "of", int(info_u.n_units.sum()))
t = units[units.target.isin(["S1", "S1-6", "none", "CG1"])].copy()
t["cond"] = np.where(t.target == "none", "sham", np.where(t.target == "CG1", "CG1 " + t.prf.astype("Int64").astype(str),
                                                          t.prf.astype("Int64").astype(str) + " Hz"))
print("All voltages:")
print(pd.crosstab([t.area], t.cond, margins=True).to_string())
t5 = t[(t.cohort == "2025") | (t.volt == 5) | (t.cond == "sham")]
print("\nVariant, 2024 files at 5 V (the paper's voltage) and sham, with every 2025 file (voltage not stated):")
print(pd.crosstab([t5.area], t5.cond, margins=True).to_string())
five = [f"{p} Hz" for p in tfus.PRFS]
print("\nSums over the five PRFs, all voltages:", pd.crosstab(t.area, t.cond)[five].sum(axis=1).to_dict(),
      "; 2024 files at 5 V:", pd.crosstab(t5.area, t5.cond)[five].sum(axis=1).to_dict())
paper = pd.DataFrame({"S1 (RSU+FSU)": [148 + 61, 312 + 54, 1050 + 247, 31 + 53, 12 + 34],
                      "POm (RSU+FSU)": [189 + 43, 233 + 17, 913 + 106, 67 + 67, 55 + 42]},
                     index=["30 Hz", "3000 Hz", "five PRFs", "sham", "CG1 at 3000 Hz"])
print("\nPaper's typed totals (Fig. 4, 5 and 6 legends), for comparison:")
print(paper.to_string())

Without types the totals can be set against the paper's RSU and FSU sums only roughly. POm units
are fewer than the paper's in every condition (110 at 3,000 Hz against 250). S1 units at 30 and
3,000 Hz outnumber the paper's when every voltage is counted (344 and 465 against 209 and 366), as
the 64-channel probes of BH594 give many units at 5, 10 and 15 V. With the 2024 files limited to
5 V they are fewer (186 and 201), and over the five PRFs fewer under both (1,243 and 756 against
1,297). Sham S1 units outnumber the paper's under both (151 against 84).

## subjects

> Simultaneous S1 and POm recordings were made in 16 rats, and S1 and auditory cortex recordings in
> a further 12 rats.

Criterion: counts of distinct rats with sessions of each recording pair under the paper's
conditions (S1-targeted tFUS, sham or CG1 control). A session is a file in the 2024 cohort and a
group of files sharing a trial table in the 2025 cohort (data check). A session holds an area when
it has a unit of that area. Targets counted as the paper's conditions: S1, sham, CG1; "S1-6" is
counted as S1-targeted under reading (a) and left out under reading (b), as nothing says what it
is. A variant counts any rat with files of both areas, simultaneous or not.

In [ ]:
def sessions_table(targets):
    rows = []
    for i in keep:
        r = info.loc[i]
        if r.target not in targets:
            continue
        areas = {unit_area(i, k) for k in range(len(files[i]["spikes"]))}
        key = (r.rat, r.trial_key) if r.cohort == "2025" else (r.rat, i)
        rows += [{"rat": r.rat, "session": key, "area": a} for a in areas]
    s = pd.DataFrame(rows).groupby(["rat", "session"]).area.agg(lambda x: set(x)).reset_index()
    return s


readings = {"(a) S1, S1-6, sham, CG1": {"S1", "S1-6", "none", "CG1"}, "(b) S1, sham, CG1": {"S1", "none", "CG1"}}
subject_counts = {}
for name, tg in readings.items():
    s = sessions_table(tg)
    pom = sorted(s[s.area.apply(lambda a: {"S1", "POm"} <= a)].rat.unique())
    ac = sorted(s[s.area.apply(lambda a: {"S1", "AC"} <= a)].rat.unique())
    subject_counts[name] = (len(pom), len(ac))
    print(f"{name}: simultaneous S1 and POm in {len(pom)} rats {pom}; S1 and AC in {len(ac)} rats {ac}; "
          f"in both groups: {sorted(set(pom) & set(ac))}")
by_rat = units.groupby("rat").area.agg(lambda x: set(x))
print("Variant, any files of both areas: S1 and POm", sorted(by_rat[by_rat.apply(lambda a: {'S1', 'POm'} <= a)].index),
      "; S1 and AC", sorted(by_rat[by_rat.apply(lambda a: {'S1', 'AC'} <= a)].index),
      "(units with more than 2,000 spikes, any target)")
print("Rats with any AC file:", sorted(info.loc[info.rec == "AC", "rat"].unique()))
s5 = info_u[(info_u.volt == 5) & (info_u.target == "S1")]
print("2024 rats with an S1-targeted 5 V session:", sorted(s5.rat.unique()))
print("Paper: 16 S1 and POm rats, 12 further S1 and AC rats.")

r946 = requests.get("https://api.dandiarchive.org/api/dandisets/000946/versions/0.250829.1249/assets/?page_size=1000",
                    timeout=60).json()
p946 = [a["path"] for a in r946["results"]]
while r946["next"]:
    r946 = requests.get(r946["next"], timeout=60).json()
    p946 += [a["path"] for a in r946["results"]]
subj946 = sorted({p.split("/")[0].replace("sub-", "") for p in p946 if not p.startswith("sub-STTC")})
dates946 = sorted({p.split("ses-")[1][:8] for p in p946 if "ses-" in p})
print(f"\nDandiset 000946 (the paper's named repository), version 0.250829.1249, asset paths only: "
      f"{len(p946)} files, rats {subj946}, session dates {dates946[0]} to {dates946[-1]}")
print("Rats of 001199 also in 000946:", sorted(set(info.rat) & set(subj946)))

**Result: partially reproduced.** Under both readings the dandiset holds simultaneous S1 and POm
recordings from 6 rats (paper 16) and S1 and AC recordings from 6 rats (paper 12). Both counts are
below the paper's and neither is above it, the claims file's rule for a partial result. Two rats
(BH648, BH658) are in both groups, where the paper's AC group is "another group of rats". The
paper's named repository, 000946, lists 19 other rats recorded from 2023 to March 2024; of the 13
rats here only BH594, an S1-only rat here, has a file there. So the dandiset may not hold the
paper's rats at all.

## fig11b-ac-responses

> Auditory cortex neurons respond during sonication to 1,500 and 3,000 Hz PRF and after sonication
> to 30, 1,500, 3,000 and 4,500 Hz, with PRF-dependent responses and the least response at 300 Hz.

Criterion: AC units with more than 2,000 spikes of the S1 and AC rats, the 200 us PD series,
normalized rate (R1, R2) in 0 to 67 and 201 to 268 ms, Kruskal-Wallis across the five PRFs at
p < 0.01, then the one-tailed rank-sum test against baseline per PRF (T1, T2) with Bonferroni's
correction, corrected p < 0.01. Bonferroni factors: B1 = 5 (the PRFs compared with baseline in a
window; 10 is shown too, for the two windows together), B2 = 15 (all pairs among the five PRFs and
the baseline). Every AC file with stimulation at S1 or S1-6 is used (all 7 rats with AC files); a
variant leaves out S1-6. The total is compared with the paper's 453.

In [ ]:
def ac_units(targets, sham=False):
    rows = []
    for i in keep:
        r = info.loc[i]
        if r.rec != "AC" or (r.target not in targets):
            continue
        sp = [s for s in files[i]["spikes"] if len(s) > tfus.MIN_SPIKES]
        if not sp:
            continue
        nr = tfus.normalized_rates(tfus.unit_counts(sp, files[i]["trial_start"], tfus.WINDOWS), tfus.WINDOWS)
        nr["rat"], nr["prf"], nr["target"], nr["file"] = r.rat, (0 if sham else r.prf), r.target, i
        rows.append(nr)
    return pd.concat(rows, ignore_index=True)


ac = ac_units({"S1", "S1-6"})
ac_sham = ac_units({"none"}, sham=True)
print("AC units with more than 2,000 spikes:", len(ac), "(paper 453); left out for no prestimulus spike:",
      int(ac["R1_0-67"].isna().sum()))
print("AC units with 2,000 spikes or fewer in these files:",
      sum(len(s) <= tfus.MIN_SPIKES for i in set(ac.file) for s in files[i]["spikes"]))
print("\nUnits per rat and PRF (paper: 30 Hz 83, 300 Hz 59 by difference, 1,500 Hz 107, 3,000 Hz 104, 4,500 Hz 100):")
print(pd.crosstab(ac.rat, ac.prf, margins=True).to_string())
print("Sham AC units (no tFUS), per rat:", ac_sham.rat.value_counts().sort_index().to_dict())


def panel(df, label):
    out = []
    for w in ("0-67", "201-268"):
        for R in ("R1", "R2"):
            g = [df[df.prf == p][f"{R}_{w}"].values for p in tfus.PRFS]
            h, pk = tfus.kruskal(g)
            for T in ("T1", "T2"):
                p = np.array([tfus.vs_baseline(df[df.prf == q], w, R, T) for q in tfus.PRFS])
                row = {"set": label, "window": w, "R": R, "T": T, "KW_H": round(h, 2), "KW_p": pk}
                for q, m, pv in zip(tfus.PRFS, [np.nanmean(x) for x in g], p):
                    row[f"mean_{q}"] = round(m, 1)
                    row[f"p_{q}"] = pv
                for B, fac in (("B1", 5), ("B1x2", 10), ("B2", 15)):
                    row[f"sig_{B}"] = ",".join(str(q) for q, pv in zip(tfus.PRFS, p) if pv * fac < 0.01)
                row["lowest"] = tfus.PRFS[int(np.nanargmin([np.nanmean(x) for x in g]))]
                out.append(row)
    return pd.DataFrame(out)


res = panel(ac, "S1 and S1-6")
fmt = res.copy()
for c in [c for c in fmt if c.startswith("p_") or c == "KW_p"]:
    fmt[c] = fmt[c].map(lambda v: f"{v:.2g}")
print("\nUncorrected one-tailed p against baseline per PRF; sig_ lists PRFs significant after each Bonferroni factor:")
print(fmt.to_string(index=False))

In [ ]:
# Reference: the same tests on AC units of the sham sessions (no tFUS), as one group per window.
rows = []
for w in ("0-67", "201-268"):
    for R in ("R1", "R2"):
        for T in ("T1", "T2"):
            rows.append({"window": w, "R": R, "T": T, "mean": round(np.nanmean(ac_sham[f"{R}_{w}"]), 1),
                         "p_increase": f"{tfus.vs_baseline(ac_sham, w, R, T):.2g}", "n": int(ac_sham[f'{R}_{w}'].notna().sum())})
print("Sham AC units:")
print(pd.DataFrame(rows).to_string(index=False))

# Variant: S1-6 left out.
res_b = panel(ac[ac.target == "S1"], "S1 only")
print("\nVariant, S1-6 sessions left out; units per PRF:", ac[ac.target == "S1"].prf.value_counts().sort_index().to_dict())
print(res_b[["window", "R", "T", "KW_H", "KW_p", "mean_30", "mean_300", "mean_1500", "mean_3000", "mean_4500",
             "sig_B1", "sig_B2", "lowest"]].to_string(index=False))

# Variant: the 6 rats whose AC file shares a session with an S1 file (BH654 left out).
res_c = panel(ac[ac.rat != "BH654"], "without BH654")
print("\nVariant, BH654 (no S1 and AC session) left out; units:", int((ac.rat != "BH654").sum()))
print(res_c[["window", "R", "T", "KW_p", "sig_B1", "sig_B2", "lowest"]].to_string(index=False))

# By rat: mean normalized rate (R1) per PRF and window, and the median, which is robust to single units.
print("\nBy rat, R1 mean (median) per PRF:")
for w in ("0-67", "201-268"):
    tab = ac.groupby(["rat", "prf"])[f"R1_{w}"].agg(lambda x: f"{x.mean():.0f} ({x.median():.0f})").unstack()
    print(f"window {w}\n", tab.to_string())
print("\nR1 0-67 ms at 3,000 Hz, largest five units:", np.round(np.sort(ac[ac.prf == 3000]["R1_0-67"].values)[-5:], 1))
print("R1 medians per PRF, 0-67:", ac.groupby("prf")["R1_0-67"].median().round(1).to_dict(),
      "; 201-268:", ac.groupby("prf")["R1_201-268"].median().round(1).to_dict())

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
for ax, w in zip(axes, ("0-67", "201-268")):
    m = ac.groupby("prf")[f"R1_{w}"].agg(["mean", "sem"])
    ax.errorbar(range(5), m["mean"], m["sem"], fmt="o")
    ax.axhline(100, color="0.6", lw=0.8)
    ax.set_xticks(range(5), [str(p) for p in tfus.PRFS]); ax.set_title(f"AC, {w} ms"); ax.set_xlabel("PRF (Hz)")
axes[0].set_ylabel("normalized rate R1 (%), mean ± SEM")
print("plotted means range %.1f to %.1f %%" % (ac.groupby("prf")[["R1_0-67", "R1_201-268"]].mean().min().min(),
                                              ac.groupby("prf")[["R1_0-67", "R1_201-268"]].mean().max().max()))

**Result: not reproduced.** There are 194 AC units with more than 2,000 spikes in 7 rats, against
the paper's 453. In the main set no Kruskal-Wallis test across the five PRFs is significant at 0.01
(p from 0.013 to 0.74); with BH654 left out the R2 tests give 0.0095 and 0.006, with every R2 mean
below 100%. Against baseline only 1,500 Hz during sonication is significant, and only under R1
and T2 (mean 105.1%, uncorrected p = 6.4e-05, below 0.01 under every factor). The paper's pattern
needs 1,500 and 3,000 Hz during sonication and 30, 1,500, 3,000 and 4,500 Hz after it. Here no
PRF is above baseline after sonication under any reading, and 300 Hz has the lowest mean only under
R1 during sonication without BH654. Per-rat R1 means lie between 88 and 127%. Under R2 every mean
is below 100%, sham included (69.8%), because R2 keeps only trials with a prestimulus spike. In the
2025 cohort the trial starts could be shown to be sonication onsets only for the BH648 4,500 Hz
session (data check), so the lack of responses may come from the trial tables and not from the neurons.

## Summary

| Claim | Outcome |
|---|---|
| subjects | partially reproduced |
| fig11b-ac-responses | not reproduced |
| fig2-target-artifact | blocked, data missing |
| 20 claims on RSUs and FSUs (Fig. 4 to 6, 10A to 10C, 11C, 11D) | blocked, data missing |
| 12 claims on the duty-cycle series (fig7-9-unit-counts, Fig. 7 to 9, 10D to 10F) | blocked, data missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. Sessions are taken as the dandiset gives them, with
"S1-6" counted as S1-targeted (a variant leaves it out). The pressure of the 2025 sessions is not
stated. AC units are all those with more than 2,000 spikes, as the Methods say, in the 7 rats with
AC files.

What was found about the dataset. The dandiset holds spike times and trial tables only: no
waveforms, so no RSU or FSU types. Its `celltype_label` column is described as RSU and FSU codes
but holds 0 or the area. Only the 200 us PD series is here, with sessions at 10 and 15 V, POm
targeting and probe variants the paper does not describe. Its 13 rats were recorded from August
2024 to July 2025, after the paper's preprint (June 2024). Of them only BH594 has a file in 000946,
the dandiset the paper names for its data. Session start times do not date the sessions of the
2025 cohort. Two files are the same recording. Several target labels are unexplained. In most 2025
sessions the spikes show no locking to the trial starts.